# GPT-2 Event-Stream Time Series Forecasting




## 0. Installation

Install the dependencies from the repository root before running this notebook:

```bash
pip install -r requirements.txt
```




## 1. Imports and environment check


In [ ]:
import os, math, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Reproducibility: this makes stochastic parts easier to compare across runs.
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2. Project paths




In [ ]:
# Project paths
# The notebook is designed to work both from a cloned GitHub repo and from Colab.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "electricity_price_daily_synthetic.csv",
    PROJECT_ROOT / "electricity_price_daily_synthetic.csv",
    Path("/content/electricity_price_daily_synthetic.csv"),  # Colab fallback
]

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ts_gpt2_tokens"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Figures dir:", FIGURES_DIR)
print("Model output dir:", MODEL_OUTPUT_DIR)


## 3. Load dataset


In [ ]:
# Load the dataset.
# Recommended GitHub layout:
#   data/electricity_price_daily_synthetic.csv
# In Colab, you may also upload the CSV manually if it is not found.
import io

existing_path = next((p for p in DATA_CANDIDATES if p.exists()), None)

if existing_path is not None:
    CSV_PATH = existing_path
    df = pd.read_csv(CSV_PATH)
    print("Loaded from path:", CSV_PATH, "shape:", df.shape)
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) > 0, "Please upload a CSV file."
        fname = next(iter(uploaded.keys()))
        df = pd.read_csv(io.BytesIO(uploaded[fname]))
        print("Loaded uploaded file:", fname, "shape:", df.shape)
    except Exception as e:
        raise RuntimeError(
            "Could not find the dataset. Put electricity_price_daily_synthetic.csv "
            "in data/ or upload it in Colab."
        ) from e

df.head()


## 4. Tokenization utilities

The time series is converted into first-order differences, then the differences are quantized into a finite set of bin tokens. The binning parameters are fitted on the training slice only to avoid validation leakage.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error

def detect_datetime_column(df: pd.DataFrame):
    """Return first column that can be parsed as datetime."""
    for col in df.columns:
        if df[col].dtype == "object" or np.issubdtype(df[col].dtype, np.datetime64):
            try:
                pd.to_datetime(df[col])
                return col
            except Exception:
                pass
    raise ValueError("No datetime-like column detected.")

def detect_numeric_column(df: pd.DataFrame):
    """Return first numeric column."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric column detected.")
    return numeric_cols[0]


def fit_bin_diffs_direct(
    diffs: np.ndarray,
    fit_end: int,
    n_bins: int = 93,
    use_outliers: bool = True,
    q_low: float = 0.25,
    q_high: float = 0.75,
    iqr_k: float = 1.5,
    standardize_diffs: bool = False,
    scaling: str = "standard",
    eps: float = 1e-8,
):
    """Fit a binning scheme on TRAIN diffs only (no leakage).

    diffs: np.ndarray of raw diffs (length N-1)
    fit_end: number of diffs to use for fitting (train slice), i.e. diffs[:fit_end]
    Returns meta dict with everything needed to transform any diffs consistently.
    """
    if n_bins < 2:
        raise ValueError("n_bins must be >= 2")
    fit_end = int(max(1, min(len(diffs), fit_end)))
    train_diffs = diffs[:fit_end].astype(float)
    train_diffs = train_diffs[np.isfinite(train_diffs)]
    if len(train_diffs) < 10:
        raise ValueError(f"Not enough train diffs to fit bins (got {len(train_diffs)}).")

    # ---- scaling (fit on train only) ----
    diff_mu = 0.0
    diff_sigma = 1.0
    train_for_bounds = train_diffs.copy()

    if standardize_diffs:
        if scaling == "standard":
            diff_mu = float(np.mean(train_diffs))
            diff_sigma = float(np.std(train_diffs) + eps)
            train_for_bounds = (train_diffs - diff_mu) / diff_sigma
        elif scaling == "mean":
            diff_mu = float(np.mean(train_diffs))
            diff_sigma = 1.0
            train_for_bounds = train_diffs / (diff_mu + eps)
        elif scaling == "minmax":
            mn = float(np.min(train_diffs))
            mx = float(np.max(train_diffs))
            diff_mu = mn
            diff_sigma = mx
            train_for_bounds = (train_diffs - mn) / (mx - mn + eps)
        else:
            raise ValueError(f"Unknown scaling method: {scaling}")

    # ---- outlier bounds (fit on train only) ----
    if use_outliers:
        d = pd.Series(train_for_bounds)
        Q1 = float(d.quantile(q_low))
        Q3 = float(d.quantile(q_high))
        IQR = Q3 - Q1
        if not np.isfinite(IQR) or abs(IQR) < eps:
            lo, hi = float(np.min(train_for_bounds)), float(np.max(train_for_bounds))
        else:
            lo = Q1 - iqr_k * IQR
            hi = Q3 + iqr_k * IQR
    else:
        lo, hi = float(np.min(train_for_bounds)), float(np.max(train_for_bounds))

    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = -1.0, 1.0

    # ---- inlier quantile edges (fit on train only) ----
    if use_outliers:
        inliers = train_for_bounds[(train_for_bounds >= lo) & (train_for_bounds <= hi)]
    else:
        inliers = train_for_bounds

    if len(inliers) < 10:
        # fallback to using all train_for_bounds
        inliers = train_for_bounds

    qs = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(inliers, qs)
    edges = np.asarray(edges, dtype=float)
    edges = np.maximum.accumulate(edges)
    if np.unique(edges).size < 3:
        edges = np.linspace(float(np.min(inliers)), float(np.max(inliers)), n_bins + 1)

    meta = {
        "n_inlier_bins": int(n_bins),                # inlier bins = 1..n_bins
        "use_outliers": bool(use_outliers),          # outlier bins: 0 (low), n_bins+1 (high)
        "q_low": float(q_low),
        "q_high": float(q_high),
        "iqr_k": float(iqr_k),
        "standardize_diffs": bool(standardize_diffs),
        "diff_scaling": scaling if standardize_diffs else None,
        "diff_mu": float(diff_mu) if standardize_diffs else None,
        "diff_sigma": float(diff_sigma) if standardize_diffs else None,
        "clip_lo": float(lo),
        "clip_hi": float(hi),
        "edges": edges,                              # length n_bins+1 for INLIERS
    }

    return meta


def transform_bin_diffs_direct(diffs: np.ndarray, meta: dict, eps: float = 1e-8):
    """Transform diffs to bin codes using a pre-fitted meta (no leakage)."""
    diffs = diffs.astype(float)

    # scale using fitted params
    x = diffs.copy()
    if meta.get("standardize_diffs", False):
        scaling = meta.get("diff_scaling", "standard")
        if scaling == "standard":
            mu = float(meta["diff_mu"])
            sigma = float(meta["diff_sigma"])
            x = (x - mu) / (sigma + eps)
        elif scaling == "mean":
            mu = float(meta["diff_mu"])
            x = x / (mu + eps)
        elif scaling == "minmax":
            mn = float(meta["diff_mu"])
            mx = float(meta["diff_sigma"])
            x = (x - mn) / (mx - mn + eps)

    lo = float(meta["clip_lo"])
    hi = float(meta["clip_hi"])
    edges = np.asarray(meta["edges"], dtype=float)
    n_bins = int(meta["n_inlier_bins"])

    # bins: 0 = low outlier, 1..n_bins = inliers, n_bins+1 = high outlier
    bin_codes = np.empty_like(x, dtype=int)

    if meta.get("use_outliers", True):
        low_mask = x < lo
        high_mask = x > hi
        inlier_mask = ~(low_mask | high_mask)

        bin_codes[low_mask] = 0
        bin_codes[high_mask] = n_bins + 1

        # digitize inlier into 0..n_bins-1 then +1 => 1..n_bins
        if inlier_mask.any():
            in_x = x[inlier_mask]
            in_x = np.clip(in_x, lo, hi)
            in_codes = np.digitize(in_x, edges[1:-1], right=False)  # 0..n_bins-1
            bin_codes[inlier_mask] = in_codes + 1
        else:
            bin_codes[:] = 0
    else:
        # all treated as inliers
        x2 = np.clip(x, lo, hi)
        in_codes = np.digitize(x2, edges[1:-1], right=False)
        bin_codes[:] = in_codes  # 0..n_bins-1

    max_bin = int(n_bins + 1) if meta.get("use_outliers", True) else int(n_bins - 1)
    return bin_codes, max_bin


def bin_diffs_direct(
    df: pd.DataFrame,
    value_col: str,
    date_col: str | None = None,
    n_bins: int = 93,
    use_outliers: bool = True,
    q_low: float = 0.25,
    q_high: float = 0.75,
    iqr_k: float = 1.5,
    standardize_diffs: bool = False,
    scaling: str = "standard",
    fit_rows: int | None = None,
):
    """Compute diffs, assign each diff to a bin code, and build bin tokens.

    IMPORTANT: binning scheme is FIT ON TRAIN ONLY (first fit_rows rows) and then applied
    to all rows. This avoids leakage from validation/test into tokenization.
    """
    df0 = df.copy()

    if date_col is not None:
        df0[date_col] = pd.to_datetime(df0[date_col])
        df0 = df0.sort_values(date_col).reset_index(drop=True)

    values = df0[value_col].astype(float).to_numpy()
    if len(values) < 3:
        raise ValueError("Need at least 3 rows to compute diffs + train.")

    diffs = np.diff(values)  # length N-1

    # Determine how many diffs are in the training portion for fitting
    if fit_rows is None:
        fit_end = len(diffs)
    else:
        fit_end = max(1, min(len(diffs), int(fit_rows) - 1))

    meta = fit_bin_diffs_direct(
        diffs=diffs,
        fit_end=fit_end,
        n_bins=n_bins,
        use_outliers=use_outliers,
        q_low=q_low,
        q_high=q_high,
        iqr_k=iqr_k,
        standardize_diffs=standardize_diffs,
        scaling=scaling,
    )

    bin_codes, max_bin = transform_bin_diffs_direct(diffs, meta)
    meta["max_bin"] = int(max_bin)
    meta["fit_rows_for_binning"] = int(fit_rows) if fit_rows is not None else None

    # Compute bin_means on TRAIN ONLY for decoding
    bin_means = {}
    for b in range(int(max_bin) + 1):
        m = diffs[:fit_end][bin_codes[:fit_end] == b].mean() if np.any(bin_codes[:fit_end] == b) else 0.0
        bin_means[b] = float(m)
    meta["bin_means"] = bin_means

    bin_tokens = [f"bin_{b}" for b in bin_codes]

    # Reconstruct using bin_means (quantization loss visualization)
    recon = [float(values[0])]
    for b in bin_codes:
        recon.append(recon[-1] + bin_means.get(int(b), 0.0))
    recon = np.array(recon)

    out = df0.copy()
    out["diff"] = np.concatenate([[np.nan], diffs])
    out["bin_code"] = np.concatenate([[np.nan], bin_codes.astype(float)])
    out["bin_token"] = np.concatenate([["start"], bin_tokens])
    out["reconstructed"] = recon

    return out, meta


def reconstruction_rmse(df_binned: pd.DataFrame, value_col: str):
    y = df_binned[value_col].astype(float).to_numpy()
    yhat = df_binned["reconstructed"].astype(float).to_numpy()
    return float(np.sqrt(mean_squared_error(y, yhat)))


## 5. Fit binning scheme and reconstruct the series

This section creates symbolic bin tokens and reconstructs the original series from train-only bin means. The reconstruction RMSE measures quantization loss before any forecasting model is used.


In [ ]:
# ---- Configure binning ----
N_BINS = 65
USE_OUTLIERS = True
Q_LOW = 0.25
Q_HIGH = 0.75
IQR_K = 1.5

# ---- NEW: diff standardization (fit on train slice only) ----
VAL_FRACTION = 0.25
STANDARDIZE_DIFFS = False
DIFF_SCALING = "standard"
FIT_ROWS = int(len(df) * (1 - VAL_FRACTION))

date_col = detect_datetime_column(df)
value_col = detect_numeric_column(df)

df_binned, meta = bin_diffs_direct(
    df,
    value_col=value_col,
    date_col=date_col,
    n_bins=N_BINS,
    use_outliers=USE_OUTLIERS,
    q_low=Q_LOW,
    q_high=Q_HIGH,
    iqr_k=IQR_K,
    standardize_diffs=STANDARDIZE_DIFFS,
    scaling=DIFF_SCALING,
    fit_rows=FIT_ROWS,
)

rmse = reconstruction_rmse(df_binned, value_col)
print("Detected date column:", date_col)
print("Detected value column:", value_col)
print("Reconstruction RMSE (quantization loss):", rmse)
print("Max bin:", meta["max_bin"])
df_binned[[date_col, value_col, "diff", "bin_code", "bin_token", "reconstructed"]].head(10)


In [ ]:
# Plot original vs reconstructed values.
# This visualizes the information loss caused by quantizing differences into bins.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_binned[date_col], df_binned[value_col], label="Original")
ax.plot(df_binned[date_col], df_binned["reconstructed"], linestyle="--", label="Reconstructed")
ax.set_title("Original vs Reconstructed from Mean Difference per Bin")
ax.set_xlabel("Date")
ax.set_ylabel(value_col.replace("_", " ").title())
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "original-vs-reconstructed.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Build custom WordLevel tokenizer

The symbolic event stream is treated like text. The vocabulary is built from the training tokens only.


In [ ]:
# Build a custom WordLevel tokenizer for the BIN tokens
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

tokens = df_binned["bin_token"].astype(str).tolist()

# Train/validation split on the token stream (bins)
n = len(tokens)
split = int(n * (1 - VAL_FRACTION))

train_tokens = tokens[:split]
val_tokens = tokens[split:]

# IMPORTANT (no leakage): build vocab from TRAIN TOKENS ONLY
special_tokens = ["[PAD]", "[BOS]", "[EOS]", "[UNK]"]
vocab_tokens = sorted(set(train_tokens))
vocab_list = special_tokens + vocab_tokens
vocab = {tok: i for i, tok in enumerate(vocab_list)}

wordlevel = WordLevel(vocab=vocab, unk_token="[UNK]")
tok = Tokenizer(wordlevel)
tok.pre_tokenizer = Whitespace()

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tok,
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
)

print("Vocab size:", tokenizer.vocab_size)
print("Train tokens:", len(train_tokens), "Val tokens:", len(val_tokens))
print("Sample encoding:", tokenizer("start bin_1 bin_2 bin_2 bin_0"))


## 7. Convert tokens to IDs and create language-modeling blocks


In [ ]:
# Build token-id streams WITHOUT joining into one huge string (better for large datasets)

UNK_ID = tokenizer.unk_token_id

# Fast mapping using the WordLevel vocab dict
token_to_id = vocab  # from the tokenizer build cell

def tokens_to_ids(tok_list):
    return [token_to_id.get(t, UNK_ID) for t in tok_list]

train_ids = tokens_to_ids(train_tokens)
val_ids   = tokens_to_ids(val_tokens)

print("Train ids:", len(train_ids), "Val ids:", len(val_ids))
print("UNK rate (train):", np.mean(np.array(train_ids) == UNK_ID))
print("UNK rate (val):  ", np.mean(np.array(val_ids) == UNK_ID))


In [ ]:
# Chunk token-id streams into fixed-length blocks for causal LM training (scales to large datasets)
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

BLOCK_SIZE = 128  # must match model.config.n_positions

def make_lm_dataset(ids, block_size=128, drop_remainder=True):
    ids = list(map(int, ids))
    if drop_remainder:
        n_blocks = len(ids) // block_size
        ids = ids[: n_blocks * block_size]
    else:
        n_blocks = int(np.ceil(len(ids) / block_size))

    input_ids = [ids[i*block_size:(i+1)*block_size] for i in range(n_blocks)]
    # pad last block if keeping remainder
    if not drop_remainder and input_ids and len(input_ids[-1]) < block_size:
        pad_id = tokenizer.pad_token_id
        last = input_ids[-1]
        input_ids[-1] = last + [pad_id] * (block_size - len(last))

    attention_mask = [[1 if t != tokenizer.pad_token_id else 0 for t in blk] for blk in input_ids]
    return Dataset.from_dict({"input_ids": input_ids, "attention_mask": attention_mask})

train_lm = make_lm_dataset(train_ids, BLOCK_SIZE, drop_remainder=True)
val_lm   = make_lm_dataset(val_ids,   BLOCK_SIZE, drop_remainder=False)

print("Train blocks:", len(train_lm), "Val blocks:", len(val_lm))

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


## 8. Define compact GPT-2 model

The model is initialized from scratch with a vocabulary that matches the custom bin-token tokenizer.


In [ ]:
# Define a small GPT‑2 from scratch (vocab matches our bin-token tokenizer)
from transformers import GPT2Config, GPT2LMHeadModel

# After tokenizer is created
tokenizer.padding_side = "right"

# When building config
config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=BLOCK_SIZE,
    n_ctx=BLOCK_SIZE,
    n_embd=384,
    n_layer=6,
    n_head=6,
    resid_pdrop=0.2,
    embd_pdrop=0.2,
    attn_pdrop=0.2,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,   # IMPORTANT
)

model = GPT2LMHeadModel(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")


In [ ]:
from torch.utils.data import DataLoader

batch = next(iter(DataLoader(train_lm, batch_size=2, collate_fn=data_collator)))
for k,v in batch.items():
    print(k, v.shape, v.dtype)

labels = batch["labels"]
print("labels == -100 ratio:", (labels == -100).float().mean().item())
print("unique label ids (sample):", torch.unique(labels[labels != -100])[:20])
print("num valid labels:", (labels != -100).sum().item())


## 9. Train the causal language model

The model learns to predict the next symbolic event token. Training/evaluation losses are logged and saved under `outputs/ts_gpt2_tokens/`.


In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback, set_seed, EarlyStoppingCallback  # ✅ add back
import matplotlib.pyplot as plt
import numpy as np
import torch
import os

# For reproducibility
set_seed(SEED)

OUTPUT_DIR = str(MODEL_OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

class LossHistoryCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.steps = []
        self.eval_steps = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        # Track training loss
        if "loss" in logs:
            self.train_losses.append(float(logs["loss"]))
            self.steps.append(int(state.global_step))

        # Track eval loss
        if "eval_loss" in logs:
            self.eval_losses.append(float(logs["eval_loss"]))
            self.eval_steps.append(int(state.global_step))

            # Live monitoring line at each evaluation point
            last_train = self.train_losses[-1] if self.train_losses else float("nan")
            print(f"[step {state.global_step}] train_loss={last_train:.4f}  eval_loss={logs['eval_loss']:.4f}")

    def plot_losses(self):
        if self.train_losses:
            plt.figure(figsize=(10,4))
            plt.plot(self.steps, self.train_losses)
            plt.xlabel("Step")
            plt.ylabel("Training loss")
            plt.grid(True, alpha=0.3)
            plt.title("Training loss")
            plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=150, bbox_inches="tight")
            plt.show()

        if self.eval_losses:
            plt.figure(figsize=(10,4))
            plt.plot(self.eval_steps, self.eval_losses)
            plt.xlabel("Step")
            plt.ylabel("Eval loss")
            plt.grid(True, alpha=0.3)
            plt.title("Eval loss")
            plt.savefig(f"{OUTPUT_DIR}/eval_loss.png", dpi=150, bbox_inches="tight")
            plt.show()

        if self.train_losses:
            print(f"Initial training loss: {self.train_losses[0]:.4f}")
            print(f"Final training loss:   {self.train_losses[-1]:.4f}")
        if self.eval_losses:
            print(f"Best eval loss:        {min(self.eval_losses):.4f}")
            print(f"Final eval loss:       {self.eval_losses[-1]:.4f}")


loss_callback = LossHistoryCallback()

NUM_EPOCHS = 80

EVAL_STEPS = 4
LOG_STEPS  = 4
SAVE_STEPS = EVAL_STEPS  # aligned so best checkpoint exists

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # overwrite_output_dir=True,

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_steps=SAVE_STEPS,
    logging_steps=LOG_STEPS,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,

    learning_rate=1e-6,
    warmup_steps=30,
    weight_decay=0.01,

    fp16=torch.cuda.is_available(),
    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_lm,
    eval_dataset=val_lm,
    data_collator=data_collator,
    # tokenizer=tokenizer,
    callbacks=[
        loss_callback,
        EarlyStoppingCallback(early_stopping_patience=3),  # ✅ stop after 3 evals without improvement
    ],
)

trainer.train()
loss_callback.plot_losses()

print("Training completed! Logs saved to:", OUTPUT_DIR)
print(f"Total training steps: {trainer.state.global_step}")
print(f"Total epochs completed: {trainer.state.epoch}")


## 10. Generate future event tokens

The trained model generates future `bin_*` tokens autoregressively.


In [ ]:
# Generate future BIN tokens
H = 60
CONTEXT_LEN = 200

max_positions = model.config.n_positions
max_prompt_len = max_positions - H
context = min(CONTEXT_LEN, max_prompt_len)

prompt_tokens = train_tokens[-context:]
prompt_text = " ".join(prompt_tokens)

inputs = tokenizer(prompt_text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

input_len = inputs["input_ids"].shape[1]
max_new = min(H, max_positions - input_len)

gen_ids = model.generate(
    **inputs,
    max_new_tokens=max_new,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.7,
)

new_ids = gen_ids[0, input_len:].tolist()
generated = tokenizer.convert_ids_to_tokens(new_ids)

generated_bins = []
for t in generated:
    if t.startswith("bin_"):
        try:
            generated_bins.append(int(t.split("_", 1)[1]))
        except Exception:
            pass

print("Prompt len:", input_len, "Generated:", len(new_ids), "Max positions:", max_positions)
print("Generated tokens:", generated[:20])
print("Generated bin codes:", generated_bins[:20], "count:", len(generated_bins))


## 11. Decode generated bins back into numerical forecasts

Generated tokens are mapped back to train-only bin means and accumulated from the last observed value.


In [ ]:
df_train = df.iloc[:split].copy()
df_train_binned, train_meta = bin_diffs_direct(
    df_train,
    value_col=value_col,
    date_col=date_col,
    n_bins=N_BINS,
    use_outliers=USE_OUTLIERS,
    q_low=Q_LOW,
    q_high=Q_HIGH,
    iqr_k=IQR_K,
    standardize_diffs=STANDARDIZE_DIFFS,
    scaling=DIFF_SCALING,
    fit_rows=len(df_train),
)
bin_means = train_meta["bin_means"]

start_value = float(df_train_binned[value_col].iloc[-1])
start_date = pd.to_datetime(df_train_binned[date_col].iloc[-1])

forecast = []
v = start_value
for b in generated_bins[:H]:
    v = v + bin_means.get(int(b), 0.0)
    forecast.append(v)


dt = pd.to_datetime(df_train_binned[date_col]).diff().median()
if pd.isna(dt) or dt == pd.Timedelta(0):
    future_dates = [start_date + pd.Timedelta(days=i+1) for i in range(len(forecast))]
else:
    future_dates = [start_date + dt*(i+1) for i in range(len(forecast))]

plt.figure(figsize=(12,4))
plt.plot(df_train_binned[date_col], df_train_binned[value_col], label="Train history")
plt.plot(future_dates, forecast, label="Forecast (generated bins)")
plt.title("Forecast decoded from predicted diff bins")
plt.legend()
plt.grid(True)
plt.show()

# Save the long-horizon generated forecast figure for the repository.
fig = plt.gcf()
try:
    fig.savefig(FIGURES_DIR / "generated-bin-forecast.png", dpi=300, bbox_inches="tight")
except Exception:
    pass


## 12. Short-horizon forecast demo

This is the main forecast visualization used for the article. Figures are saved to `results/figures/`.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# =========================
# STYLE CONFIG
# =========================
BG = "#000000"
FG = "#f3f4f6"
GRID = (1, 1, 1, 0.10)

BLUE = "#67c6ff"
ORANGE = "#f4a261"
GOLD = "#ffd166"
MUTED = "#b8bec9"

OUTPUT_IMAGE_DIR = str(FIGURES_DIR)
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": FG,
    "axes.labelcolor": FG,
    "axes.edgecolor": (1, 1, 1, 0.18),
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.titleweight": "bold",
    "axes.titlesize": 18,
    "axes.labelsize": 13,
    "font.size": 12,
    "legend.frameon": False,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, color=GRID, linewidth=1)
    for spine in ax.spines.values():
        spine.set_color((1, 1, 1, 0.18))
        spine.set_linewidth(1.0)
    ax.tick_params(colors=MUTED, labelsize=11)
    ax.xaxis.label.set_color(FG)
    ax.yaxis.label.set_color(FG)
    ax.title.set_color(FG)

# =========================
# DURING GENERATION (FORECASTING)
# =========================
WINDOW = 20
HORIZON = 10
USE_SAMPLING = True

t0 = split - 1  # last index of TRAIN portion in the full token stream

# Build history (for plotting) from the same window we prompt with
context_len = min(WINDOW, model.config.n_positions - HORIZON, t0 + 1)
prompt_start = t0 - context_len + 1

hist_dates = pd.to_datetime(df_binned[date_col].iloc[prompt_start:t0+1])
hist_vals  = df_binned[value_col].iloc[prompt_start:t0+1].astype(float).to_numpy()

# Prepare prompt tokens
prompt_tokens = tokens[prompt_start : t0 + 1]
prompt_text = " ".join(prompt_tokens)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

gen_kwargs = {
    "max_new_tokens": HORIZON,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
}
gen_kwargs.update(
    {"do_sample": True, "top_k": 30, "top_p": 0.9, "temperature": 0.7}
    if USE_SAMPLING else
    {"do_sample": False}
)

with torch.no_grad():
    gen_ids = model.generate(**inputs, **gen_kwargs)

new_ids = gen_ids[0, inputs["input_ids"].shape[1]:]
pred_token_strs = tokenizer.convert_ids_to_tokens(new_ids)

# Filter out special tokens AND 'start' if it appears
pred_token_strs = [
    t for t in pred_token_strs
    if t not in ("[PAD]", "[BOS]", "[EOS]", "[UNK]", "start")
]

# Enforce exactly HORIZON tokens (pad/truncate)
if len(pred_token_strs) < HORIZON:
    last_tok = prompt_tokens[-1] if prompt_tokens else "bin_0"
    pred_token_strs += [last_tok] * (HORIZON - len(pred_token_strs))
pred_token_strs = pred_token_strs[:HORIZON]

print("Predicted bin tokens:", pred_token_strs)

pred_bins = [
    int(t.split("_", 1)[1]) if t.startswith("bin_") else 0
    for t in pred_token_strs
]

# Decode forecasted values by accumulating bin_means diffs
start_value = float(df_binned[value_col].iloc[t0])
v = start_value
forecast_vals = []
for b in pred_bins:
    v += float(meta["bin_means"].get(int(b), 0.0))
    forecast_vals.append(v)
forecast_vals = np.array(forecast_vals)

# Compare to actual future values (if available)
real_future_print = df_binned["bin_token"].iloc[t0+1 : t0+1+HORIZON]
print(real_future_print)

real_future = df_binned[value_col].iloc[t0+1 : t0+1+HORIZON].astype(float).to_numpy()

if len(real_future) == len(forecast_vals):
    mae = float(np.mean(np.abs(real_future - forecast_vals)))
    rmse = float(np.sqrt(np.mean((real_future - forecast_vals) ** 2)))
    print(f"MAE={mae:.4f} | RMSE={rmse:.4f} | window={context_len} horizon={HORIZON}")
else:
    mae = float("nan")
    rmse = float("nan")
    print("Not enough future points to compute MAE/RMSE for this horizon.")

# =========================
# BUILD DATES SO LINES CONNECT AT t0
# =========================
t0_date = pd.to_datetime(df_binned[date_col].iloc[t0])
future_dates = pd.to_datetime(df_binned[date_col].iloc[t0+1 : t0+1+HORIZON])

plot_dates = pd.Index([t0_date]).append(pd.Index(future_dates))

real_series = (
    np.concatenate([[start_value], real_future])
    if len(real_future)
    else np.array([start_value])
)
pred_series = np.concatenate([[start_value], forecast_vals])

# =========================
# PLOT IN YOUR STYLE
# =========================
fig, ax = plt.subplots(figsize=(14, 5.5))

# History
ax.plot(
    hist_dates,
    hist_vals,
    color=BLUE,
    linewidth=2.4,
    label=f"History (last {len(hist_vals)})"
)

# Forecast start
ax.axvline(
    t0_date,
    color=GOLD,
    linestyle="--",
    linewidth=1.8,
    alpha=0.95,
    label="Forecast start"
)

# Real future
if len(real_future):
    ax.plot(
        plot_dates,
        real_series,
        color=BLUE,
        linewidth=2.8,
        alpha=0.95,
        label=f"Real next {len(real_future)}"
    )

# Predicted future
ax.plot(
    plot_dates,
    pred_series,
    color=ORANGE,
    linewidth=2.8,
    linestyle="--",
    alpha=0.98,
    label=f"Predicted next {HORIZON}"
)

# Optional highlight of forecasted points
ax.scatter(
    plot_dates[1:],
    pred_series[1:],
    color=ORANGE,
    s=28,
    zorder=3
)

if len(real_future):
    ax.scatter(
        plot_dates[1:],
        real_series[1:],
        color=BLUE,
        s=24,
        zorder=3
    )

title = (
    f"Forecast Demo | MAE = {mae:.3f}, RMSE = {rmse:.3f}"
    if np.isfinite(rmse)
    else "Forecast Demo"
)
ax.set_title(title, pad=14)
ax.set_xlabel("Date")
ax.set_ylabel(value_col.replace("_", " ").title())

style_ax(ax)

leg = ax.legend(loc="upper left", fontsize=11)
for t in leg.get_texts():
    t.set_color(FG)

fig.tight_layout()

# Show
plt.show()

# Save for article
forecast_path = os.path.join(OUTPUT_IMAGE_DIR, "gpt2-forecast-demo.png")
fig.savefig(forecast_path, dpi=220, bbox_inches="tight")
print("Saved:", forecast_path)
plt.close(fig)


## 13. Baseline comparison with Darts

This section compares the GPT-style event-stream forecast against classical baselines: Naive Seasonal, Theta, and Exponential Smoothing.


In [ ]:
# Optional baseline dependency
# The final section compares the GPT-style forecast with Darts baselines.
# Install dependencies before running the notebook:
#   pip install -r requirements.txt
# If you only want to train/generate the GPT-style model, you can skip the baseline section.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NaiveSeasonal, Theta, ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =========================
# STYLE CONFIG
# =========================
BG = "#000000"
FG = "#f3f4f6"
GRID = (1, 1, 1, 0.10)

BLUE = "#67c6ff"       # Real
ORANGE = "#f4a261"     # GPT-2
GOLD = "#ffd166"       # Theta
SILVER = "#cbd5e1"     # Naive
ICE = "#8ecae6"        # ExpSmooth
MUTED = "#b8bec9"

OUTPUT_IMAGE_DIR = str(FIGURES_DIR)
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": FG,
    "axes.labelcolor": FG,
    "axes.edgecolor": (1, 1, 1, 0.18),
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "axes.titleweight": "bold",
    "axes.titlesize": 17,
    "axes.labelsize": 13,
    "font.size": 12,
    "legend.frameon": False,
})

def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, color=GRID, linewidth=1)
    for spine in ax.spines.values():
        spine.set_color((1, 1, 1, 0.18))
        spine.set_linewidth(1.0)
    ax.tick_params(colors=MUTED, labelsize=11)
    ax.xaxis.label.set_color(FG)
    ax.yaxis.label.set_color(FG)
    ax.title.set_color(FG)

def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

# =========================
# FIT BASELINES
# =========================
series = TimeSeries.from_series(df[value_col])
train_series = series[:t0+1]

naive_model = NaiveSeasonal(K=10)
theta_model = Theta(theta=2)
es_model = ExponentialSmoothing()

naive_model.fit(train_series)
theta_model.fit(train_series)
es_model.fit(train_series)

darts_naive_vals = naive_model.predict(HORIZON).values().flatten()
darts_theta_vals = theta_model.predict(HORIZON).values().flatten()
darts_es_vals = es_model.predict(HORIZON).values().flatten()

# =========================
# METRICS
# =========================
mae_gpt, rmse_gpt = compute_metrics(real_future, forecast_vals)
mae_naive, rmse_naive = compute_metrics(real_future, darts_naive_vals)
mae_theta, rmse_theta = compute_metrics(real_future, darts_theta_vals)
mae_es, rmse_es = compute_metrics(real_future, darts_es_vals)

print("\n=== Model Comparison ===")
print(f"GPT-2        | MAE={mae_gpt:.4f} | RMSE={rmse_gpt:.4f}")
print(f"Naive        | MAE={mae_naive:.4f} | RMSE={rmse_naive:.4f}")
print(f"Theta        | MAE={mae_theta:.4f} | RMSE={rmse_theta:.4f}")
print(f"ExpSmooth    | MAE={mae_es:.4f} | RMSE={rmse_es:.4f}")

# =========================
# BUILD SERIES FOR PLOTTING
# =========================
start_value = float(df[value_col].iloc[t0])

series_dict = {
    "Real": real_series,
    "GPT-2": pred_series,
    "Naive": np.concatenate([[start_value], darts_naive_vals]),
    "Theta": np.concatenate([[start_value], darts_theta_vals]),
    "ExpSmooth": np.concatenate([[start_value], darts_es_vals]),
}

# Choose what to display
SELECT_MODELS = ["Real", "GPT-2", "Naive"]

# Color map
color_map = {
    "Real": BLUE,
    "GPT-2": ORANGE,
    "Naive": SILVER,
    "Theta": GOLD,
    "ExpSmooth": ICE,
}

# Line style map
style_map = {
    "Real": "-",
    "GPT-2": "--",
    "Naive": "--",
    "Theta": "--",
    "ExpSmooth": "--",
}

width_map = {
    "Real": 3.0,
    "GPT-2": 2.8,
    "Naive": 2.2,
    "Theta": 2.2,
    "ExpSmooth": 2.2,
}

# =========================
# PLOT
# =========================
fig, ax = plt.subplots(figsize=(14.5, 6))

# History
ax.plot(
    hist_dates,
    hist_vals,
    color=MUTED,
    linewidth=2.2,
    alpha=0.9,
    label=f"History (last {len(hist_vals)})"
)

# Forecast start
ax.axvline(
    t0_date,
    color=FG,
    linestyle="--",
    linewidth=1.4,
    alpha=0.55,
    label="Forecast start"
)

# Plot selected models
for name in SELECT_MODELS:
    if name in series_dict:
        ax.plot(
            plot_dates,
            series_dict[name],
            linestyle=style_map[name],
            linewidth=width_map[name],
            color=color_map[name],
            alpha=0.98 if name in ("Real", "GPT-2") else 0.92,
            label=name
        )

        # highlight future points
        ax.scatter(
            plot_dates[1:],
            series_dict[name][1:],
            color=color_map[name],
            s=22 if name in ("Real", "GPT-2") else 16,
            alpha=0.95,
            zorder=3
        )

# Dynamic title
title_metrics = []
if "GPT-2" in SELECT_MODELS:
    title_metrics.append(f"GPT-2 RMSE={rmse_gpt:.3f}")
if "Naive" in SELECT_MODELS:
    title_metrics.append(f"Naive RMSE={rmse_naive:.3f}")
if "Theta" in SELECT_MODELS:
    title_metrics.append(f"Theta RMSE={rmse_theta:.3f}")
if "ExpSmooth" in SELECT_MODELS:
    title_metrics.append(f"ES RMSE={rmse_es:.3f}")

title = f"Forecast Comparison | Horizon = {HORIZON}"
if title_metrics:
    title += "\n" + " | ".join(title_metrics)

ax.set_title(title, pad=14)
ax.set_xlabel("Date")
ax.set_ylabel(value_col.replace("_", " ").title())

style_ax(ax)

leg = ax.legend(loc="upper left", fontsize=10, ncol=2)
for t in leg.get_texts():
    t.set_color(FG)

fig.tight_layout()

# Show
plt.show()

# Save for article
comparison_path = os.path.join(OUTPUT_IMAGE_DIR, "forecast-comparison-baselines.png")
fig.savefig(comparison_path, dpi=220, bbox_inches="tight")
print("Saved:", comparison_path)

plt.close(fig)
